In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
from functools import reduce
import uuid
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []
skipped = []

for table_name in SOURCE_TABLES:
    source_path = f'{SOURCE_LH_ABFSS}/{SOURCE_SCHEMA}/{table_name}'
    bronze_path = f'{BRONZE_LH_ABFSS}/{BRONZE_SCHEMA}/{table_name}'
    try:
        df = spark.read.format('delta').load(source_path)
        (df.withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
           .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
           .withColumn('_INGESTED_AT', F.current_timestamp())
           .withColumn('_IS_DELETED', F.lit(False))
           .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(bronze_path))
        results.append({'table': table_name, 'rows': df.count()})
        print(f'{table_name}: loaded')
    except Exception as exc:
        skipped.append({'table': table_name, 'error': str(exc)})
        print(f'{table_name}: skipped ({exc})')

print(f'Bronze complete: {len(results)} loaded, {len(skipped)} skipped')